In [11]:
import os
import json
import re
import numpy as np
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack, csr_matrix


# ===== Класс для выделения числовых признаков =====
class NumericExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.keys = set()
        for row in X:
            if "numeric_parameters" in row:
                self.keys.update(row["numeric_parameters"].keys())
        self.keys = sorted(list(self.keys))
        return self

    def transform(self, X):
        result = []
        for row in X:
            values = []
            for key in self.keys:
                val = None
                if "numeric_parameters" in row and key in row["numeric_parameters"]:
                    val = row["numeric_parameters"][key].get("value")
                values.append(val if val is not None else np.nan)
            result.append(values)
        return np.nan_to_num(np.array(result))


# ===== Папка с JSON =====
data_dir = "epi_rash/preprocessed_medical_results"

# ===== Загрузка файлов =====
records = []
for fname in os.listdir(data_dir):
    if fname.endswith(".json"):
        with open(os.path.join(data_dir, fname), "r", encoding="utf-8") as f:
            try:
                records.append(json.load(f))
            except Exception as e:
                print(f"⚠️ Ошибка чтения {fname}: {e}")

print(f"Загружено файлов: {len(records)}")

# ===== Формируем X и y =====
X_records = []
y = []
for rec in records:
    diagnosis = rec.get("diagnosis", {}).get("diagnosis_text")
    if diagnosis is not None and diagnosis != "":
        y.append(diagnosis)
        X_records.append(rec)

print("После фильтрации с диагнозами:", len(X_records))

# ===== Подготовка текстов =====
texts = []
for rec in X_records:
    complaints = rec.get("complaints", {}).get("processed_text", "")
    disease_history = rec.get("disease_history", {}).get("processed_text", "")
    text = (complaints or "") + " " + (disease_history or "")
    text = re.sub(r"[^а-яА-Яa-zA-Z0-9 ]", " ", text)  # очищаем
    text = text.strip()
    if not text:
        text = "нет_данных"
    texts.append(text)

print("Пример текстов:", texts[:3])

# ===== TF-IDF =====
if all(t == "нет_данных" for t in texts):
    X_text = csr_matrix((len(texts), 1))  # фиктивный признак
else:
    vectorizer = TfidfVectorizer(max_features=500)
    X_text = vectorizer.fit_transform(texts)

# ===== Числовые признаки =====
num_extractor = NumericExtractor()
X_num = num_extractor.fit_transform(X_records)

# ===== Комбинация признаков =====
X = hstack([X_text, X_num]).tocsr()  # теперь можно индексировать

# ===== Обработка редких классов =====
counts = Counter(y)
valid_idx = [i for i, label in enumerate(y) if counts[label] > 1]

if len(valid_idx) < len(y):
    print(f"⚠️ Удалены классы с одним примером: {len(y) - len(valid_idx)}")
    X = X[valid_idx]
    y = [y[i] for i in valid_idx]

# ===== Обучение =====
if len(set(y)) < 2 or len(y) < 5:
    print("⚠️ Недостаточно данных для train/test, обучаем на всех примерах")
    clf = RandomForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X, y)
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    clf = RandomForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    print("\n=== Отчет по качеству ===")
    print(classification_report(y_test, y_pred, digits=4))


Загружено файлов: 2013
После фильтрации с диагнозами: 2010
Пример текстов: ['средний тяжести  лихорадка темперaтура 39 6 с  кашель мoкротой  одышка небольшой физическoть нагрузке  общий слабость  ломота теле  ноябрь 2020 перенести коронавирусный инфекцию  covid 19  лабораторный подтвержденную  39  потеря вкус обоняния  настоящий ухудшение начало март 2021 г  когда появиться боль горле  покашливание  07 03 2021 находиться аршане  быть интенсивный физический нагрузка  09 03 2021 ухудшилось  ломота теле  повышение температура 39 6  сухой кашель  пцр sars cov2 отрицательный  сохраняться лихорадка  кашель  появиться одышка  доставить при мный отделение гбуз иокб  выполнить мскт ангиография ла инфильтрация нижний доля оба легких  двухсторонний гидроторакс  медиастинальный лимфоаденопатия  гидроперикард  тэла определяется  сколиоз грудной отдел позвоночника  госпитализировать отделение пульмонологии   расти ла развиваться лася соответственно пол возрасту  сопутствовать хронический тонзилит  в

c:\Users\PC2\Desktop\Klukin\Epikriz_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\PC2\Desktop\Klukin\Epikriz_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\PC2\Desktop\Klukin\Epikriz_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,